# Frame Zero · train the reported detector

One notebook, the whole result: the main model, the ablations that support the
novelty claim, and the results table that goes in the report.

**Push your local fixes first** — this clones the repo:

```
git add -A && git commit -m "T4 config, manifest, resume, consistency flag" && git push
```

### The plan

| Run | What it is | Why it exists | T4 | L4/A100 |
|---|---|---|---|---|
| **A** | full model, every source | the headline result | ~6 h | ~2 h |
| **B** | `--consistency-weight 0` | evidence the novelty does something | ~6 h | ~2 h |
| **C** | `--backbone mesonet` | the cited network vs a modern one | ~1 h | ~20 m |
| D | `--no-srm` | optional third ablation row | ~4 h | ~1.5 h |

Run A alone gives you a working detector. A and B together give you a
**defensible contribution** — that pairing is what turns the consistency loss
from an assertion into a measured effect. Run C is cheap and fills the
comparison section.

Checkpoints go to Drive and every training cell auto-resumes, so a disconnect
costs at most one epoch. Re-run the cell and it continues.

## 1 · Setup

In [ ]:
import torch, subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU"

# Rough speed factor vs a T4, for the time estimates further down.
SPEED = 1.0
for name, mult in [("A100", 4.0), ("L4", 2.5), ("V100", 1.8), ("P100", 1.2)]:
    if name in gpu:
        SPEED = mult
print(f"\nassuming ~{SPEED:.1f}x T4 speed for the estimates below")

In [ ]:
# Training needs none of the face-detection stack - the crops are already
# extracted. Skipping facenet-pytorch and mediapipe also avoids their torch
# pins fighting Colab's torch.
!pip install -q timm scikit-learn pandas tqdm

import os
REPO = "/content/deepfake_system"
if not os.path.exists(REPO):
    !git clone -q https://github.com/sailessawesome-ui/deepfake_system.git {REPO}
else:
    !cd {REPO} && git pull -q

WORK = f"{REPO}/deepfake_system"
RUNS = "/content/drive/MyDrive/deepfake_runs"
print("working dir:", WORK)

## 2 · Data

All four archives are listed below whether or not you have them yet. Anything
missing is reported and skipped, so when DF40 arrives you just re-run this cell
and the manifest cell — nothing to edit.

The cell prints exactly what will be trained on and what is held out, so you can
confirm it before committing hours.

**WildDeepfake is never trained on.** `config.DATA.holdout_sources` excludes it,
so it lands in the `holdout` split and becomes your unseen-source number. That
is the only honest generalisation figure you will have; training on it destroys
it permanently.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil, time

# Keys must match the folder names in config.DATA.sources.
ARCHIVES = {
    "FF++":          "/content/drive/MyDrive/FF++.zip",
    "celebdf_faces": "/content/drive/MyDrive/celebdf_faces.zip",
    "df40_frames":   "/content/drive/MyDrive/df40_frames.zip",   # AI-generated faces
    "wild":          "/content/drive/MyDrive/wild.zip",          # HOLDOUT, never trained
}

# All four are listed even before they exist. Anything missing is reported
# and skipped, so when DF40 lands you re-run this cell and the manifest
# cell - no editing at 2am.

DATA_ROOT = "/content/data"
os.makedirs(DATA_ROOT, exist_ok=True)

for folder, zip_path in ARCHIVES.items():
    dest = os.path.join(DATA_ROOT, folder)
    if os.path.exists(dest) and os.listdir(dest):
        print(f"{folder}: already extracted")
        continue
    if not os.path.exists(zip_path):
        print(f"{folder}: MISSING at {zip_path} - skipping")
        continue
    t0 = time.time()
    local = f"/content/{os.path.basename(zip_path)}"
    print(f"{folder}: copying...", flush=True)
    shutil.copy(zip_path, local)
    print(f"{folder}: unzipping...", flush=True)
    os.makedirs(dest, exist_ok=True)
    !unzip -q {local} -d {dest}
    os.remove(local)
    print(f"{folder}: {time.time()-t0:.0f}s")

print()
!df -h /content | tail -1
print()
have = [f for f in ARCHIVES if os.path.exists(os.path.join(DATA_ROOT, f))]
missing = [f for f in ARCHIVES if f not in have]

print("TRAINED ON     :", ", ".join(h for h in have if h != "wild") or "nothing")
print("HELD OUT       :", "wild" if "wild" in have else "- (no unseen-source number)")
if missing:
    print("NOT PRESENT    :", ", ".join(missing))

if "df40_frames" not in have:
    print()
    print("NOTE: DF40 is absent. FF++ and Celeb-DF are both face-SWAP datasets,")
    print("      so the detector will have no coverage of AI-generated faces and")
    print("      the unseen_method split will be empty. Add DF40 before the")
    print("      final run, or state the limitation in the report.")

## 3 · Manifest

Splits are taken from the `train`/`val`/`test` folders already inside the
archives. That matters for Celeb-DF, whose folder names are hashes: identity
cannot be recovered from them, so identity grouping would silently degrade into
a plain video split. `--ignore-path-splits` forces identity grouping instead.

In [ ]:
!cd {WORK} && python -m data.build_manifest --root /content/data --dry-run

In [ ]:
!cd {WORK} && python -m data.build_manifest --root /content/data --out /content/manifest.csv

In [ ]:
import pandas as pd

df = pd.read_csv("/content/manifest.csv")
print(f"{len(df):,} frames | {df.video_id.nunique():,} videos")
print()

per = df.groupby(["split", "label"]).video_id.nunique().unstack(fill_value=0)
per.columns = ["real", "fake"][:len(per.columns)]
print("VIDEOS PER SPLIT"); print(per); print()
print("SOURCE x SPLIT")
print(df.groupby(["source", "split"]).video_id.nunique().unstack(fill_value=0))
print()

leaks = df.groupby("video_id").split.nunique()
leaks = leaks[leaks > 1]
print("=" * 58)
print("FAIL: %d videos in two splits" % len(leaks) if len(leaks)
      else "PASS: every video sits in exactly one split")

straddle = df[df.source == "ffpp"].groupby("identity").split.nunique()
straddle = straddle[straddle > 1]
print(f"FF++ identities spanning splits: {len(straddle)}"
      + ("  <-- optimistic numbers" if len(straddle) else "  (clean)"))

for s in ("train", "val", "test"):
    if s not in per.index:
        print(f"WARNING: no '{s}' split")
    elif (per.loc[s] == 0).any():
        print(f"WARNING: split '{s}' is missing a class")

n_train = df[df.split == "train"].video_id.nunique()
print(f"\ntrain videos: {n_train:,}  ->  {n_train // 8:,} steps per epoch")
print("=" * 58)

## 4 · Benchmark before committing

Times a dozen real steps. Tells you whether the batch fits and how many hours
Run A will take, in about a minute.

In [ ]:
import sys, time, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

sys.path.insert(0, WORK)
from config import DATA, MODEL, TRAIN
from data.dataset import ClipDataset, load_manifest, make_sampler
from models.net import build_model, param_groups

print(f"img {DATA.img_size} | clip_len {DATA.clip_len} | batch {TRAIN.batch_size}"
      f" | srm {MODEL.use_srm}")
print(f"images per step: {TRAIN.batch_size * DATA.clip_len * 2}")

tv = load_manifest("/content/manifest.csv", splits=("train",))
bds = ClipDataset(tv, train=True, clips_per_video=2, paired=True)
bld = DataLoader(bds, batch_size=TRAIN.batch_size, sampler=make_sampler(tv),
                 num_workers=TRAIN.num_workers, pin_memory=True, drop_last=True)

m = build_model(MODEL).cuda()
o = torch.optim.AdamW(param_groups(m, TRAIN.lr, TRAIN.backbone_lr_mult,
                                   TRAIN.weight_decay))
sc = torch.amp.GradScaler("cuda")
torch.cuda.reset_peak_memory_stats()

times = []
m.train()
for i, b in enumerate(bld):
    if i >= 15:
        break
    torch.cuda.synchronize(); t0 = time.time()
    with torch.autocast("cuda"):
        lc, _ = m(b["clean"].cuda(non_blocking=True))
        ld_, _ = m(b["degraded"].cuda(non_blocking=True))
        y = b["label"].cuda(non_blocking=True)
        loss = (F.binary_cross_entropy_with_logits(lc, y) +
                F.binary_cross_entropy_with_logits(ld_, y))
    sc.scale(loss).backward(); sc.step(o); sc.update()
    o.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    if i >= 3:
        times.append(time.time() - t0)

sps = sorted(times)[len(times)//2]
steps = len(bld)
peak = torch.cuda.max_memory_allocated() / 1024**3
cap = torch.cuda.get_device_properties(0).total_memory / 1024**3

print()
print("=" * 58)
print(f"median step     : {sps:.2f}s")
print(f"steps per epoch : {steps:,}")
print(f"one epoch       : {steps*sps/60:.0f} min")
print(f"Run A ({TRAIN.epochs} ep)   : {steps*sps*TRAIN.epochs/3600:.1f} h")
print(f"A + B + C       : ~{steps*sps*TRAIN.epochs/3600*2.2:.1f} h")
print()
print(f"peak VRAM       : {peak:.1f} / {cap:.1f} GB", end="  ")
print("TIGHT - drop batch_size to 6" if peak > cap*0.9 else "comfortable")
print("=" * 58)

del m, o, sc, bld, bds
torch.cuda.empty_cache()

## 5 · Run A — the detector

This is the model that ships. Re-run the cell after any disconnect; it resumes
from the last completed epoch with the optimizer, LR schedule and EMA intact.

In [ ]:
import os
RUN_A = f"{RUNS}/A_full"
os.makedirs(RUN_A, exist_ok=True)
resume_a = f"--resume {RUN_A}/last.pt" if os.path.exists(f"{RUN_A}/last.pt") else ""
cmd_a = (f"cd {WORK} && python train.py --manifest /content/manifest.csv "
         f"--out {RUN_A} {resume_a}")
print(cmd_a)

In [ ]:
!{cmd_a}

`val f1` should climb fast for two or three epochs then flatten. If it sits near
0.5 after three epochs the labels are wrong — go back to the manifest check.

A val F1 above ~0.97 is expected and **is not your headline number**. These
splits are in-distribution. The numbers that carry weight come next.

## 6 · The evaluation battery

`evaluate.py` calibrates the threshold and temperature on **val**, never on
test, then scores every video twice — clean and messenger-degraded — and breaks
the result down by source and by generator family.

Splits that do not exist in your manifest are skipped automatically.

In [ ]:
import pandas as pd, subprocess

splits = set(pd.read_csv("/content/manifest.csv").split.unique())
targets = [s for s in ("test", "unseen_method", "holdout") if s in splits]
print("evaluating:", ", ".join(targets))
for s in ("unseen_method", "holdout"):
    if s not in splits:
        why = ("needs DF40" if s == "unseen_method" else "needs WildDeepfake")
        print(f"  skipping '{s}' - not in manifest ({why})")

In [ ]:
for split in targets:
    print("\n" + "#" * 62)
    print(f"# {split}")
    print("#" * 62, flush=True)
    !cd {WORK} && python evaluate.py --checkpoint {RUN_A}/best.pt --manifest /content/manifest.csv --split {split}

## 7 · The results table

Assembles the report JSONs into the table that goes in your write-up, as
markdown you can paste directly.

In [ ]:
import json, os

ROW_LABEL = {
    ("test", "clean"):           "Held-out videos, same datasets",
    ("test", "degraded"):        "Same, messenger-transcoded",
    ("unseen_method", "clean"):  "Unseen generator family",
    ("unseen_method", "degraded"): "Unseen generator, transcoded",
    ("holdout", "clean"):        "Unseen dataset (WildDeepfake)",
    ("holdout", "degraded"):     "Unseen dataset, transcoded",
}

rows = []
for split in targets:
    path = f"{RUN_A}/report_{split}.json"
    if not os.path.exists(path):
        continue
    rep = json.load(open(path))
    for cond in ("clean", "degraded"):
        m = rep.get("by_condition", {}).get(cond)
        if not m:
            continue
        rows.append((ROW_LABEL.get((split, cond), f"{split} / {cond}"),
                     m["n"], m["acc"], m["f1"],
                     m.get("auc"), m["precision"], m["recall"]))

print("| Condition | n | Acc | F1 | AUC | Prec | Recall |")
print("|---|---:|---:|---:|---:|---:|---:|")
for label, n, acc, f1, auc, pr, rc in rows:
    a = f"{auc:.3f}" if auc is not None else "—"
    print(f"| {label} | {n} | {acc:.3f} | {f1:.3f} | {a} | {pr:.3f} | {rc:.3f} |")

# Weakest generator families - the honest part of the report.
tp = f"{RUN_A}/report_test.json"
if os.path.exists(tp):
    bm = json.load(open(tp)).get("by_method", {})
    if bm:
        print()
        print("**Per generator family (weakest first)**")
        print()
        print("| Generator | n | F1 | Recall |")
        print("|---|---:|---:|---:|")
        for k, v in sorted(bm.items(), key=lambda kv: kv[1]["f1"])[:12]:
            print(f"| {k} | {v['n']} | {v['f1']:.3f} | {v['recall']:.3f} |")

## 8 · Run B — the consistency ablation

**This is the run that turns your novelty claim into evidence.** Identical data,
identical protocol, one term removed: `--consistency-weight 0`.

What you are looking for is the *degraded* row dropping further in B than in A.
That is the claim — the agreement term stops the model using codec artifacts as
a shortcut, so it survives transcoding. If the gap does not appear, that is a
real finding too, and reporting it honestly is worth more than quietly dropping
the experiment.

In [ ]:
RUN_B = f"{RUNS}/B_no_consistency"
os.makedirs(RUN_B, exist_ok=True)
resume_b = f"--resume {RUN_B}/last.pt" if os.path.exists(f"{RUN_B}/last.pt") else ""
cmd_b = (f"cd {WORK} && python train.py --manifest /content/manifest.csv "
         f"--out {RUN_B} --consistency-weight 0 {resume_b}")
print(cmd_b)

In [ ]:
!{cmd_b}

In [ ]:
!cd {WORK} && python evaluate.py --checkpoint {RUN_B}/best.pt --manifest /content/manifest.csv --split test

## 9 · Run C — MesoInception-4 baseline

The network cited in IR section 1.4: 28k parameters against EfficientNetV2-S's
21M. It will lose badly on accuracy, and that is the point — same data, same
protocol, so the comparison is fair. Cheap: roughly an hour on a T4.

In [ ]:
RUN_C = f"{RUNS}/C_mesonet"
os.makedirs(RUN_C, exist_ok=True)
resume_c = f"--resume {RUN_C}/last.pt" if os.path.exists(f"{RUN_C}/last.pt") else ""
cmd_c = (f"cd {WORK} && python train.py --manifest /content/manifest.csv "
         f"--out {RUN_C} --backbone mesonet --no-srm {resume_c}")
print(cmd_c)

In [ ]:
!{cmd_c}

In [ ]:
!cd {WORK} && python evaluate.py --checkpoint {RUN_C}/best.pt --manifest /content/manifest.csv --split test

## 10 · Ablation table

The comparison section of your report. The **degraded** column is the one that
carries the argument.

In [ ]:
import json, os

VARIANTS = [
    ("A · full model",            RUN_A),
    ("B · no consistency loss",   RUN_B),
    ("C · MesoInception-4",       RUN_C),
]

print("| Variant | Backbone | SRM | Consist. | Clean F1 | Degraded F1 | Drop |")
print("|---|---|:-:|:-:|---:|---:|---:|")
for name, run in VARIANTS:
    ck, rp = f"{run}/best.pt", f"{run}/report_test.json"
    if not (os.path.exists(ck) and os.path.exists(rp)):
        print(f"| {name} | _not run yet_ | | | | | |")
        continue
    import torch
    cfg = torch.load(ck, map_location="cpu", weights_only=False).get("config", {})
    rep = json.load(open(rp)).get("by_condition", {})
    clean = rep.get("clean", {}).get("f1")
    deg = rep.get("degraded", {}).get("f1")
    drop = (clean - deg) if (clean is not None and deg is not None) else None
    print(f"| {name} | {cfg.get('backbone','?')} "
          f"| {'yes' if cfg.get('use_srm') else 'no'} "
          f"| {cfg.get('consistency_weight','?')} "
          f"| {clean:.3f} | {deg:.3f} | {drop:.3f} |"
          if clean is not None else f"| {name} | incomplete | | | | | |")

print()
print("Read it as: a SMALLER drop in A than in B is the consistency loss")
print("doing its job. That single comparison is your contribution.")

## 11 · Export

Copy `best.pt` and `calibration.json` into `runs/v1/` beside the app and restart
the server. The header chip flips from `baseline · no checkpoint` to
`model · tf_efficientnetv2_s`; nothing else changes.

In [ ]:
import os, shutil

EXPORT = "/content/drive/MyDrive/deepfake_export"
os.makedirs(EXPORT, exist_ok=True)

for label, run in VARIANTS:
    tag = label.split(" ")[0]
    for f in ("best.pt", "calibration.json", "history.json",
              "report_test.json", "report_unseen_method.json",
              "report_holdout.json"):
        src = os.path.join(run, f)
        if os.path.exists(src):
            dst = os.path.join(EXPORT, f"{tag}_{f}" if tag != "A" else f)
            shutil.copy(src, dst)
            print(f"{os.path.basename(dst):34s} {os.path.getsize(src)/1024**2:7.1f} MB")

print()
print(f"Drive folder: {EXPORT}")
print("Locally:  cp best.pt calibration.json deepfake_system/runs/v1/")
print("          cd deepfake_system && ./run.sh")

---

## Reading your own results honestly

Row 1 will be 97–99%. Every published method gets that, because train and test
share a dataset. It demonstrates the pipeline works; it is not a finding.

Rows 2–4 are the report. The framing that survives questioning:

> The system reaches X% accuracy and F1 at video level on the held-out
> in-distribution test set, retains Y% under messenger transcoding, and Z% on
> generator families never seen in training. The gap is characterised rather
> than hidden.

An unqualified "99% on any video" takes one examiner with a phone to disprove.

**Two limitations to state before anyone finds them:**

1. Celeb-DF folder names are hashes here, so identity cannot be recovered.
   Those splits are video-disjoint but not provably identity-disjoint. FF++ is
   identity-clean (`008_990` → `008`).
2. Without DF40, `unseen_method` is empty and you have no unseen-generator row
   at all. FF++ and Celeb-DF are both face-swap; nothing in either covers
   diffusion or fusion.

## Troubleshooting

**OOM.** Lower `TRAIN.batch_size` to 6 or 4, or `DATA.img_size` to 192, and
raise `accum_steps` to keep the effective batch near 32. A step costs
`2 × batch × clip_len` images of activations, so clip_len drives memory as hard
as batch size does.

**Disconnected.** Re-run setup (cells 2–3), the unzip cell (skips what is
already there), then the training cell. It resumes.

**val f1 stuck at 0.5.** Labels are wrong. Re-run the manifest sanity check.

**GPU idle, slow steps.** The dataloader is starving it. `degrade_clip` is about
70% of the CPU cost per item; raise `num_workers` if you have the cores.